# Evaluate + Compare

Loads all saved depth maps from Drive, computes proxy metrics, shows comparison grids.

**Runtime:** CPU is fine — no GPU needed

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np
import cv2
import pandas as pd
from IPython.display import display, Image as IPImage

DATASET_IMAGES = '/content/drive/MyDrive/Corn Seed Dataset/test/images'
DATASET_LABELS = '/content/drive/MyDrive/Corn Seed Dataset/test/labels'
SAVE_DIR       = '/content/drive/MyDrive/Corn Seed Dataset/depth_comparison_outputs'
SAMPLE_FILE    = os.path.join(SAVE_DIR, 'sample_images.txt')

SAMPLES = [l.strip() for l in open(SAMPLE_FILE) if l.strip()]
print(f'Loaded {len(SAMPLES)} samples')

MODELS = ['depth_pro', 'moge2', 'depthfm', 'pixel_perfect', 'vggt', 'depth_anything_v3']
available = [m for m in MODELS if os.path.isdir(os.path.join(SAVE_DIR, m, 'depth'))]
print(f'Models with outputs: {available}')

def load_bboxes(stem):
    path = os.path.join(DATASET_LABELS, f'{stem}.txt')
    if not os.path.exists(path): return []
    boxes = []
    for line in open(path):
        p = line.split()
        if len(p) < 5: continue
        cx,cy,w,h = float(p[1]),float(p[2]),float(p[3]),float(p[4])
        boxes.append((max(0,int((cx-w/2)*512)), max(0,int((cy-h/2)*512)),
                      min(512,int((cx+w/2)*512)), min(512,int((cy+h/2)*512))))
    return boxes

## Compute metrics

In [ ]:
!pip install -q pandas

def score_model(name):
    edges, contrasts, smooths = [], [], []
    for img_name in SAMPLES:
        stem = img_name.split('.')[0]
        dp = os.path.join(SAVE_DIR, name, 'depth', f'{stem}.png')
        if not os.path.exists(dp): continue
        depth = cv2.imread(dp, -1).astype(np.float32)
        if depth is None or depth.max() == depth.min(): continue
        dn = (depth - depth.min()) / (depth.max() - depth.min())
        rgb_g = cv2.imread(os.path.join(DATASET_IMAGES, img_name), cv2.IMREAD_GRAYSCALE)
        for x1,y1,x2,y2 in load_bboxes(stem):
            if x2<=x1 or y2<=y1: continue
            re = cv2.Canny(rgb_g[y1:y2,x1:x2], 50, 150)
            de = cv2.Canny((dn[y1:y2,x1:x2]*255).astype(np.uint8), 50, 150)
            dd = cv2.dilate(de, np.ones((7,7), np.uint8))
            if re.sum() > 0:
                edges.append(float(((re>0) & (dd>0)).sum()) / max(1, (re>0).sum()))
            mask = np.zeros((512,512), bool); mask[y1:y2,x1:x2] = True
            contrasts.append(abs(dn[mask].mean() - dn[~mask].mean()))
            smooths.append(float(dn[y1:y2,x1:x2].var()))
    n = len(contrasts)
    if n == 0: return {'n': 0}
    return {'n': n,
            'edge_align': float(np.mean(edges)) if edges else 0,
            'contrast':   float(np.mean(contrasts)),
            'smoothness': float(np.mean(smooths))}

results = {m: score_model(m) for m in available}
with open(os.path.join(SAVE_DIR, 'results_summary.json'), 'w') as f:
    json.dump(results, f, indent=2)

rows = []
for m, r in results.items():
    if r.get('n', 0) == 0:
        rows.append({'model': m, 'n': 0, 'edge_align': 'N/A', 'contrast': 'N/A', 'smoothness': 'N/A'})
    else:
        rows.append({'model': m, 'n': r['n'],
                     'edge_align':  f"{r['edge_align']:.4f}",
                     'contrast':    f"{r['contrast']:.4f}",
                     'smoothness':  f"{r['smoothness']:.6f}"})
display(pd.DataFrame(rows).set_index('model'))

scored = {m: r for m, r in results.items() if r.get('n', 0) > 0}
if scored:
    winner = max(scored, key=lambda m: scored[m]['edge_align'])
    print(f'\nBest edge alignment: {winner}  ({scored[winner]["edge_align"]:.4f})')

## Comparison grids (first 10 images)

In [ ]:
for img_name in SAMPLES[:10]:
    stem = img_name.split('.')[0]
    rgb = cv2.resize(cv2.imread(os.path.join(DATASET_IMAGES, img_name)), (200, 200))
    cv2.putText(rgb, 'RGB', (3, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1)
    panels = [rgb]
    for m in available:
        vp = os.path.join(SAVE_DIR, m, 'vis', f'{stem}.png')
        p = cv2.resize(cv2.imread(vp), (200,200)) if os.path.exists(vp) else np.zeros((200,200,3), np.uint8)
        cv2.putText(p, m[:14], (3, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255,255,255), 1)
        panels.append(p)
    grid_path = f'/content/{stem}_grid.jpg'
    cv2.imwrite(grid_path, np.hstack(panels))
    display(IPImage(filename=grid_path))